# Extract Semantic Model Metadata
Extract granular metadata from Semantic Models using **semantic-link-labs** and write it to **managed Delta tables**.

Primary output tables:
- `lineage_semantic_models`
- `lineage_semantic_model_measures`
- `lineage_semantic_model_columns`
- `lineage_semantic_model_relationships`
- `lineage_semantic_model_functions`
- `lineage_semantic_model_extraction_log`

## 1. Imports

**Note**: Packages loaded from Fabric Environment (no installation needed).

In [ ]:
# All packages available from attached Fabric Environment
import json
from datetime import datetime
from typing import Any, Dict, List

from pyspark.sql import SparkSession  # type: ignore[reportMissingImports]
from sempy.fabric import list_datasets  # type: ignore[reportMissingImports]
from sempy_labs.tom import connect_semantic_model  # type: ignore[reportMissingImports]

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.builder.getOrCreate()

print("✅ Imports successful (semantic-link-labs + spark)")

## 2. Configuration
Parameters passed from custom item via Spark Livy API.

In [ ]:
# These will be provided as parameters when triggered via Livy
# For testing, set manually:
WORKSPACE_IDS = ["your-workspace-id"]  # Can extract from multiple workspaces
LAKEHOUSE_ID = "your-lakehouse-id"

print(f"Extracting from {len(WORKSPACE_IDS)} workspace(s)")

## 3. Extract Semantic Models

In [ ]:
def _serialize_value(value: Any) -> Any:
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, datetime):
        return value.isoformat()
    return json.dumps(value, default=str)


def _sanitize_column_name(name: str) -> str:
    sanitized_chars = []
    for ch in str(name).strip():
        if ch.isalnum() or ch == "_":
            sanitized_chars.append(ch.lower())
        else:
            sanitized_chars.append("_")

    sanitized = "".join(sanitized_chars)
    while "__" in sanitized:
        sanitized = sanitized.replace("__", "_")
    sanitized = sanitized.strip("_")

    if not sanitized:
        sanitized = "col"
    if sanitized[0].isdigit():
        sanitized = f"col_{sanitized}"

    return sanitized


def _add_sanitized_row_fields(prepared_row: Dict[str, Any], raw_row: Dict[str, Any]) -> None:
    used_names = set(prepared_row.keys())

    for raw_key, raw_value in raw_row.items():
        base_name = _sanitize_column_name(raw_key)
        final_name = base_name
        suffix = 2

        while final_name in used_names:
            final_name = f"{base_name}_{suffix}"
            suffix += 1

        prepared_row[final_name] = _serialize_value(raw_value)
        used_names.add(final_name)


def _prepare_rows(
    rows: List[Dict[str, Any]],
    workspace_id: str,
    model_id: str,
    model_name: str,
    extraction_timestamp: str
 ) -> List[Dict[str, Any]]:
    prepared_rows = []

    for row in rows:
        prepared_row = {
            "workspace_id": workspace_id,
            "model_id": model_id,
            "model_name": model_name,
            "extraction_timestamp": extraction_timestamp
        }
        _add_sanitized_row_fields(prepared_row, row)
        prepared_rows.append(prepared_row)

    return prepared_rows


def _delta_table_exists(table_name: str) -> bool:
    return bool(spark.catalog.tableExists(table_name))


def _load_delta_table(table_name: str):
    return spark.table(table_name)


def _drop_table_if_exists(table_name: str) -> None:
    if _delta_table_exists(table_name):
        spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")


def _prepare_table_for_write(table_name: str, mode: str) -> None:
    if mode == "overwrite":
        _drop_table_if_exists(table_name)


def _append_to_delta(table_name: str, rows: List[Dict[str, Any]], mode: str = "append") -> int:
    if not rows:
        return 0

    _prepare_table_for_write(table_name, mode)
    df = spark.createDataFrame(rows)

    writer = (
        df.write
        .format("delta")
        .option("mergeSchema", "true")
    )

    if mode == "overwrite":
        writer.mode("overwrite").saveAsTable(table_name)
    else:
        writer.mode("append").saveAsTable(table_name)

    return len(rows)


def _build_model_row(
    model_id: str,
    workspace_id: str,
    model_name: str,
    extraction_timestamp: str,
    duration: float,
    model_info_row: Dict[str, Any],
    counts: Dict[str, int]
) -> Dict[str, Any]:
    row = {
        "workspace_id": workspace_id,
        "model_id": model_id,
        "model_name": model_name,
        "extraction_timestamp": extraction_timestamp,
        "extraction_method": "sempy_labs.tom.connect_semantic_model",
        "extraction_duration": duration,
        "raw_model_info": json.dumps(model_info_row, default=str)
    }

    _add_sanitized_row_fields(row, model_info_row)

    for key, value in counts.items():
        row[f"count_{key}"] = value

    return row


def _build_log_rows(
    workspace_id: str,
    model_id: str,
    model_name: str,
    extraction_timestamp: str,
    status: str,
    error_message: str | None = None
) -> List[Dict[str, Any]]:
    if not error_message:
        return []

    return [{
        "workspace_id": workspace_id,
        "model_id": model_id,
        "model_name": model_name,
        "extraction_timestamp": extraction_timestamp,
        "status": status,
        "message": error_message
    }]


def extract_semantic_model(model_id: str, workspace_id: str, model_info_row: Dict[str, Any] | None = None) -> dict:
    """Extract semantic model metadata and return rows grouped by target Delta table."""
    print(f"\n📊 Extracting Semantic Model: {model_id}")

    extraction_start = datetime.now()
    model_info_row = model_info_row or {}

    try:
        print("  └─ Connecting to semantic model...")
        with connect_semantic_model(dataset=model_id, workspace=workspace_id, readonly=True) as tom:
            measures = []
            for measure in tom.all_measures():
                measures.append({
                    "Name": measure.Name,
                    "Expression": measure.Expression,
                    "Description": measure.Description if hasattr(measure, "Description") else None,
                    "Table": measure.Parent.Name,
                    "FormatString": measure.FormatString if hasattr(measure, "FormatString") else None
                })

            columns = []
            for column in tom.all_columns():
                columns.append({
                    "Name": column.Name,
                    "DataType": str(column.DataType),
                    "Description": column.Description if hasattr(column, "Description") else None,
                    "Table": column.Parent.Name,
                    "SourceColumn": column.SourceColumn if hasattr(column, "SourceColumn") else None
                })

            relationships = []
            for rel in tom.model.Model.Relationships:
                relationships.append({
                    "Name": rel.Name if hasattr(rel, "Name") else f"{rel.FromTable.Name}_{rel.ToTable.Name}",
                    "FromTable": rel.FromTable.Name,
                    "FromColumn": rel.FromColumn.Name,
                    "ToTable": rel.ToTable.Name,
                    "ToColumn": rel.ToColumn.Name,
                    "CrossFilteringBehavior": str(rel.CrossFilteringBehavior)
                })

            functions = []
            for function in tom.all_functions():
                functions.append({
                    "Name": function.Name,
                    "Expression": function.Expression,
                    "Description": function.Description if hasattr(function, "Description") else None
                })

        extraction_end = datetime.now()
        extraction_timestamp = extraction_end.isoformat()
        duration = (extraction_end - extraction_start).total_seconds()

        model_name = (
            model_info_row.get("Dataset Name")
            or model_info_row.get("Name")
            or model_info_row.get("name")
            or str(model_id)
        )

        counts = {
            "measures": len(measures),
            "columns": len(columns),
            "relationships": len(relationships),
            "functions": len(functions)
        }

        print(
            f"  ✅ Extracted: {counts['measures']} measures, {counts['columns']} columns, "
            f"{counts['relationships']} relationships"
        )

        table_rows = {
            "lineage_semantic_models": [_build_model_row(
                model_id=model_id,
                workspace_id=workspace_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp,
                duration=duration,
                model_info_row=model_info_row,
                counts=counts
            )],
            "lineage_semantic_model_measures": _prepare_rows(
                rows=measures,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_columns": _prepare_rows(
                rows=columns,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_relationships": _prepare_rows(
                rows=relationships,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_functions": _prepare_rows(
                rows=functions,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            )
        }

        return {
            "artifactId": model_id,
            "artifactType": "SemanticModel",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "modelName": model_name,
            "status": "success",
            "counts": counts,
            "tableRows": table_rows,
            "metadata": {
                "extractionDuration": duration,
                "status": "success"
            }
        }

    except Exception as e:
        extraction_end = datetime.now()
        extraction_timestamp = extraction_end.isoformat()
        duration = (extraction_end - extraction_start).total_seconds()
        model_name = (
            model_info_row.get("Dataset Name")
            or model_info_row.get("Name")
            or model_info_row.get("name")
            or str(model_id)
        )

        error_message = str(e)
        print(f"  ❌ Error: {error_message}")

        return {
            "artifactId": model_id,
            "artifactType": "SemanticModel",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "modelName": model_name,
            "status": "error",
            "counts": {},
            "tableRows": {
                "lineage_semantic_model_extraction_log": _build_log_rows(
                    workspace_id=workspace_id,
                    model_id=model_id,
                    model_name=model_name,
                    extraction_timestamp=extraction_timestamp,
                    status="error",
                    error_message=error_message
                )
            },
            "metadata": {
                "status": "error",
                "errorMessage": error_message,
                "extractionDuration": duration
            }
        }


print("✅ Semantic model extraction helpers are ready")
print("   • output is written as managed Delta tables")

## 4. Process All Workspaces

In [ ]:
# Get all Semantic Models from target workspaces and write extraction to Delta tables
all_results = []
table_write_counts: Dict[str, int] = {}

EXPECTED_TABLES = [
    "lineage_semantic_models",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_relationships",
    "lineage_semantic_model_functions",
    "lineage_semantic_model_extraction_log"
]

WRITE_MODE = "append"  # set to "overwrite" for clean reruns

if WRITE_MODE == "overwrite":
    print("🧹 WRITE_MODE=overwrite: dropping existing managed tables before extraction")
    for table_name in EXPECTED_TABLES:
        _drop_table_if_exists(table_name)

for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing workspace: {workspace_id}")

    try:
        datasets_df = list_datasets(workspace=workspace_id)

        if datasets_df.empty:
            print("  No Semantic Models found in workspace")
            continue

        models = datasets_df.to_dict("records")
        print(f"  Found {len(models)} Semantic Model(s)")

        for model in models:
            model_id = model.get("Dataset ID") or model.get("id")
            model_name = model.get("Dataset Name") or model.get("name") or model_id

            if not model_id:
                print(f"  ⚠️  Skipping model '{model_name}' - no valid ID")
                continue

            print(f"  Processing: {model_name}")
            result = extract_semantic_model(
                model_id=model_id,
                workspace_id=workspace_id,
                model_info_row=model
            )
            all_results.append(result)

            write_mode = WRITE_MODE
            for table_name, rows in result.get("tableRows", {}).items():
                written_count = _append_to_delta(table_name, rows, mode=write_mode)
                if written_count:
                    table_write_counts[table_name] = table_write_counts.get(table_name, 0) + written_count
                    print(f"    └─ Wrote {written_count} row(s) to {table_name}")

            if write_mode == "overwrite":
                write_mode = "append"

    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")

print(f"\n✅ Extraction complete: {len(all_results)} Semantic Model(s) processed")
print("\nDelta rows written:")
for table_name in sorted(table_write_counts.keys()):
    print(f"   {table_name}: {table_write_counts[table_name]}")

## 5. Summary

In [ ]:
# Display summary
success_count = sum(1 for r in all_results if r.get("status") == "success")
error_count = len(all_results) - success_count

print("\n" + "="*50)
print("EXTRACTION SUMMARY (SEMANTIC MODELS + DELTA)")
print("="*50)
print(f"Total Semantic Models: {len(all_results)}")
print(f"✅ Successful: {success_count}")
print(f"❌ Errors: {error_count}")
print("\nDelta tables written:")
for table_name in sorted(table_write_counts.keys()):
    print(f"- {table_name}: {table_write_counts[table_name]} row(s)")
print("="*50)

---
# 🧪 Testing Guide

## How to Test Semantic Model Extraction

### Option 1: Test Single Model (Recommended)
Use the test cell below to extract ONE model before running full workspace extraction.

### Option 2: Test Full Extraction
Update configuration (cell 4) and run all cells for complete workspace extraction.

## 🧪 Test: Extract Single Model

In [ ]:
# TEST: Extract a single Semantic Model and write to Delta tables
TEST_WORKSPACE_ID = "your-workspace-id"  # Get from workspace URL
TEST_MODEL_ID = "your-model-id"          # Get from Semantic Model properties

if TEST_WORKSPACE_ID != "your-workspace-id" and TEST_MODEL_ID != "your-model-id":
    print("🧪 Testing single model extraction...\n")

    test_result = extract_semantic_model(
        model_id=TEST_MODEL_ID,
        workspace_id=TEST_WORKSPACE_ID,
        model_info_row={}
    )

    if test_result.get("status") == "success":
        counts = test_result.get("counts", {})
        print("\n✅ SUCCESS!")
        print(f"   Measures: {counts.get('measures', 0)}")
        print(f"   Columns: {counts.get('columns', 0)}")
        print(f"   Relationships: {counts.get('relationships', 0)}")
        print(f"   Functions: {counts.get('functions', 0)}")
        print(f"   Duration: {test_result.get('metadata', {}).get('extractionDuration', 0):.2f}s")

        test_write_counts = {}
        test_mode = "overwrite"
        for table_name, rows in test_result.get("tableRows", {}).items():
            written_count = _append_to_delta(table_name, rows, mode=test_mode)
            if written_count:
                test_write_counts[table_name] = written_count
                print(f"   Wrote {written_count} row(s) to {table_name}")
            if test_mode == "overwrite":
                test_mode = "append"

    else:
        print(f"\n❌ FAILED: {test_result.get('metadata', {}).get('errorMessage', 'Unknown error')}")

else:
    print("⚠️  Update TEST_WORKSPACE_ID and TEST_MODEL_ID to run test")
    print("\nHow to get IDs:")
    print("1. Workspace ID: From workspace URL (last GUID)")
    print("   Example: https://app.fabric.microsoft.com/groups/{workspace-id}/list")
    print("\n2. Model ID: Open Semantic Model → Settings → Copy object ID")

## 🧪 Verify Saved Data

In [ ]:
# Inspect the Delta tables written by this notebook
table_names = [
    "lineage_semantic_models",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_relationships",
    "lineage_semantic_model_functions",
    "lineage_semantic_model_extraction_log"
],

for table_name in table_names:
    try:
        if not _delta_table_exists(table_name):
            print(f"⚠️  {table_name}: table does not exist yet")
            continue

        row_count = _load_delta_table(table_name).count()
        print(f"📊 {table_name}: {row_count} row(s)")
        if row_count > 0:
            display(_load_delta_table(table_name).limit(5))
    except Exception as e:
        print(f"⚠️  {table_name}: not available yet ({e})")

---
## 📋 Troubleshooting

### Common Issues:

**1. "ModuleNotFoundError: No module named 'sempy'"**
- Attach Fabric Environment with semantic-link packages (top toolbar → Environment dropdown)
- Verify environment is published (not Draft status)
- Run `00_setup.ipynb` to verify environment configuration

**2. "401 Unauthorized" or "403 Forbidden"**
- Verify workspace permissions (Contributor or higher required)
- Check model permissions (viewer access minimum)
- Ensure correct workspace/model IDs

**3. "Empty DataFrames"**
- Model might have no measures/columns (unlikely)
- Check if semantic-link-labs supports the model version
- Try different model

**4. "Timeout" or "Performance Issues"**
- Large models (1000+ measures) take 30-60 seconds
- Consider extracting fewer workspaces at once
- Use cell 11 to test single model first

### Expected Performance:
- Small model (< 50 measures): 5-10 seconds
- Medium model (50-500 measures): 10-30 seconds
- Large model (> 500 measures): 30-60 seconds

### Next Steps After Testing:
1. ✅ Verify extraction works with test model
2. Update `WORKSPACE_IDS` in cell 4 with target workspaces
3. Run full extraction (cells 4-8)
4. Check lakehouse for results
5. Proceed to `02_extract_reports.ipynb`